In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


## Data prerequisite

This workflow uses PubChem assay exports. If the selected AID is not present under `repos/abstractgraph-graphicalizer/data-local/PUBCHEM/`, run the [PubChem preparation notebook](../../../abstractgraph-graphicalizer/notebooks/data_preparation/prepare_pubchem_assay.ipynb) for that AID first. It downloads the active/inactive SDF pair and writes a manifest.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from abstractgraph.utils import plot_dataset_method_bars, plot_pareto
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

# NeuralGraphEstimator Demo

This notebook compares a classical graph estimator with the hierarchical neural estimator. The neural model decomposes each input graph into an AbstractGraph and encodes mapped subgraphs in node or edge mode.

In [ ]:
import numpy as np
import networkx as nx
import torch
from sklearn.metrics import accuracy_score

from abstractgraph.vectorize import AbstractGraphNodeTransformer
from abstractgraph_ml.neural import NeuralGraphEstimator
from abstractgraph.operators import *

# Suppress PyTorch Transformer nested tensor warning in this notebook
import warnings
warnings.filterwarnings(
    'ignore',
    message=r'.*enable_nested_tensor is True, but self.use_nested_tensor is False.*',
    category=UserWarning,
    module='torch.nn.modules.transformer'
)

## UMAP visualization of test embeddings

In [ ]:
import time
from sklearn.metrics import roc_auc_score, average_precision_score
from abstractgraph.utils import plot_embedding_2d
def perf(neural_graph_estimator, graphs, targets):    
    # Train metrics
    t_pred0 = time.perf_counter()
    pred = neural_graph_estimator.predict(graphs)
    try:
        proba = neural_graph_estimator.predict_proba(graphs)
    except Exception:
        proba = None
    t_pred1 = time.perf_counter()
    pred_time = t_pred1 - t_pred0
    acc = accuracy_score(targets, pred)
    errors = int((pred != targets).sum())
    if proba is not None and proba.ndim == 2 and proba.shape[1] > 1:
        if proba.shape[1] == 2:
            auc = roc_auc_score(targets, proba[:, 1])
            ap = average_precision_score(targets, proba[:, 1])
        else:
            auc = roc_auc_score(targets, proba, average='macro', multi_class='ovr')
            ap = average_precision_score(targets, proba, average='macro')
    else:
        auc, ap = None, None
    return acc, auc, ap, errors, pred_time

def _compute_embeddings(graph_estimator, graphs):
    try:
        import umap.umap_ as umap

        embs = graph_estimator.transform(graphs)
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r".*n_jobs value .* overridden .* by setting random_state.*",
                category=UserWarning,
                module="umap.umap_",
            )
            Z = umap.UMAP(n_components=2, random_state=42, init="random").fit_transform(embs)
        return Z
    except Exception:
        return None
    

def plot(graph_estimator, graphs, targets, graphs_test, targets_test):
    alpha=.65
    size = 6
    embeddings = _compute_embeddings(graph_estimator, graphs)
    fig, axes = plt.subplots(1, 3, figsize=(3.1*size, size), constrained_layout=True)
    _ = plot_embedding_2d(graph_estimator, graphs, targets, title_prefix="Train embeddings", mode="scatter", alpha=alpha, ax=axes[0], show=False, Z=embeddings)
    _ = plot_embedding_2d(graph_estimator, graphs, targets, title_prefix="Train embeddings", mode="knn_class_union", k=5, alpha=alpha, z=1, ax=axes[1], show=False, show_instances=False, Z=embeddings)
    _ = plot_embedding_2d(graph_estimator, graphs, targets, title_prefix="Train embeddings", mode="knn_class_union", k=11, alpha=alpha, z=1, ax=axes[2], show=False, show_instances=False, Z=embeddings)
    plt.show()

    embeddings = _compute_embeddings(graph_estimator, graphs_test)
    fig, axes = plt.subplots(1, 3, figsize=(3.1*size, size), constrained_layout=True)
    _ = plot_embedding_2d(graph_estimator, graphs_test, targets_test, title_prefix="Test embeddings", mode="scatter", alpha=alpha, ax=axes[0], show=False, Z=embeddings)
    _ = plot_embedding_2d(graph_estimator, graphs_test, targets_test, title_prefix="Test embeddings", mode="knn_class_union", k=5, alpha=alpha, z=1, ax=axes[1], show=False, show_instances=False, Z=embeddings)
    _ = plot_embedding_2d(graph_estimator, graphs_test, targets_test, title_prefix="Test embeddings", mode="knn_class_union", k=11, alpha=alpha, z=1, ax=axes[2], show=False, show_instances=False, Z=embeddings)
    plt.show()

def print_perf(header_str, acc, auc, ap, errors, fit_time, pred_time, n_instances):
    n_instances_per_second = n_instances / (fit_time+pred_time)
    print(f"{header_str}: accuracy={acc:.3f}, roc_auc={(auc if auc is not None else float('nan')):.3f}, avg_precision={(ap if ap is not None else float('nan')):.3f}, errors={errors}, fit_time={fit_time:.2f}s, pred_time={pred_time:.2f}s, total_time={(fit_time+pred_time):.2f}s, n_instances_per_second={n_instances_per_second:.1f}  ")

## Dataset info & examples

In [ ]:
from abstractgraph_graphicalizer.chem import PubChemAssayLoader

loader = PubChemAssayLoader(on_error="skip")
all_assay_ids = ['2631','624249','651741','588350','463230','492952']
assay_ids = all_assay_ids[-1:]
#assay_ids = ['624249'] #bundled-safe assay example
datasets = []
for assay_id in assay_ids:
    original_graphs, original_targets = loader.load(assay_id)
    print(f'AID{assay_id}  #graphs: {len(original_graphs)}')
    datasets.append((assay_id, original_graphs, original_targets))

In [ ]:
nbits = 11

def make_nsppk(radius=1, distance=3, connector=1):
    if connector == 0:
        return add(
            union_of_shortest_paths(length=distance),
            compose(combination(number_of_elements=2, distance=(0, distance)), neighborhood(radius=(0, radius))),
        )
    if connector == 1:
        return compose(combination(number_of_elements=2, distance=(0, distance)), neighborhood(radius=(0, radius)))
    raise ValueError(f"connector {connector} is not allowed")

df = make_nsppk(radius=1, distance=4, connector=1)

In [ ]:
nbits = 11
df = neighborhood(radius=(2, 2))

In [ ]:
from abstractgraph.display import display_decomposition_graph

print('Graph decomposition')
display_decomposition_graph(df)

In [ ]:
from abstractgraph.vectorize import AbstractGraphTransformer
transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=df,
    return_dense=True,
    n_jobs=-1,
)

from sklearn.ensemble import RandomForestClassifier
from abstractgraph_ml.estimators import GraphEstimator
graph_estimator = GraphEstimator(
    transformer=transformer,
    estimator=RandomForestClassifier(random_state=0, n_estimators=300, n_jobs=-1),
    postprocessor=None,
    n_selected_features=500,
)

In [ ]:
from abstractgraph_ml.neural import NeuralGraphEstimator

# Switch to "edge" to create one local token per mapped-subgraph edge.
element_mode = "node"
neural_graph_estimator = NeuralGraphEstimator(
    decomposition_function=df,
    nbits=nbits,
    element_mode=element_mode,
    mode="classification",
    # Compact per-subgraph Transformer.
    local_d_model=32,
    local_nhead=4,
    local_num_layers=1,
    local_dim_feedforward=64,
    # Compact across-interpretation-node Transformer.
    global_d_model=32,
    global_nhead=4,
    global_num_layers=1,
    global_dim_feedforward=64,
    dropout=0.3,
    embedding_dropout=0.3,
    # Supervised training.
    epochs=200,
    batch_size=64,
    val_split=0.1,
    early_stopping_patience=30,
    seed=4213,
    validation_seed=4213,
    lr=1e-3,
    weight_decay=1e-2,
    lr_scheduler_patience=8,
    lr_scheduler_factor=0.5,
    lr_scheduler_min_lr=1e-5,
    verbose=True,
)

In [ ]:
from sklearn.model_selection import train_test_split
import time

assay_id, graphs, targets = datasets[0]
n_instances = len(graphs)
targets = np.asarray(targets)
# Keep both classes represented in the train and test partitions.
graphs_tr, graphs_te, targets_tr, targets_te = train_test_split(
    graphs,
    targets,
    test_size=0.2,
    random_state=0,
    stratify=targets,
)

print('_' * 100)
print(f'assay_id:{assay_id}   #graphs:{n_instances}   #train:{len(graphs_tr)}   #test:{len(graphs_te)}')

print('Decomposition Graph Kernel Estimator')
t_fit0 = time.perf_counter()
graph_estimator.fit(graphs_tr, targets_tr)
t_fit1 = time.perf_counter()
fit_time = t_fit1 - t_fit0
acc_te, auc_te, ap_te, errors_te, pred_time_te = perf(graph_estimator, graphs_te, targets_te)
print_perf('Test', acc_te, auc_te, ap_te, errors_te, fit_time, pred_time_te, len(graphs_te))
plot(graph_estimator, graphs_tr, targets_tr, graphs_te, targets_te)

print(f'Hierarchical Neural Graph Estimator ({element_mode} mode)')
t_fit0 = time.perf_counter()
neural_graph_estimator.fit(graphs_tr, targets_tr)
t_fit1 = time.perf_counter()
fit_time = t_fit1 - t_fit0
acc_te, auc_te, ap_te, errors_te, pred_time_te = perf(neural_graph_estimator, graphs_te, targets_te)
print_perf('Test', acc_te, auc_te, ap_te, errors_te, fit_time, pred_time_te, len(graphs_te))
plot(neural_graph_estimator, graphs_tr, targets_tr, graphs_te, targets_te)


## Standalone neural experiments

All runs below used the same stratified train/test split. Neural models used validation based early stopping; the graph kernel is a reference baseline and its predictions are never blended with a neural model.

| Model | Configuration | Test accuracy | ROC AUC |
| --- | --- | ---: | ---: |
| Hierarchical Transformer | Node mode, width 32, best single run | 0.732 | 0.771 |
| Hierarchical Transformer ensemble | Five node seeds, validation selected threshold | 0.723 | 0.790 |
| Hierarchical Transformer | Edge mode, width 32 | 0.725 | 0.766 |
| Hierarchical Transformer | Edge mode rerun, width 32, same split, 200 epoch cap, patience 30; MPS; val-loss checkpoint at epoch 16 | 0.710 | 0.766 |
| Hierarchical Transformer | Edge mode, width 64 | 0.706 | 0.754 |
| Direct graph neural network | Three model GIN ensemble | 0.684 | 0.748 |
| Fingerprint MLP | 2,048 input features, width 256 | 0.732 | 0.777 |
| Neural factorization machine | Rank 32, five full data refits, validation threshold | 0.749 | 0.814 |
| Neural factorization machine | Rank 32, five fold threshold calibration | 0.743 | 0.814 |
| Neural factorization machine | Rank 8, stronger regularization | 0.697 | 0.764 |
| Neural factorization machine | Added molecule descriptors | 0.712 | 0.765 |
| Neural factorization network | Nonlinear pairwise interaction head | 0.725 | 0.760 |
| Hierarchical Transformer | Multi assay pretraining and target fine tuning | 0.688 | 0.758 |
| Fingerprint MLP | 500 validation selected features, width 128 | 0.708 | 0.774 |
| Hierarchical Transformer | Node mode, nbits 14 | 0.671 | 0.692 |
| Hierarchical Transformer | Node mode, nbits 10 | 0.693 | 0.718 |
| Hierarchical Transformer | Latent mixup, alpha 0.2 | 0.725 | 0.776 |
| Hierarchical Transformer ensemble | Node model plus latent mixup model | 0.723 | 0.778 |
| Hierarchical Transformer | Added mapped atom bond context | 0.723 | 0.779 |
| Direct graph neural network | Added valence, bond counts and ring flags | 0.682 | 0.742 |
| Hierarchical Transformer | Class 0 loss weight 1.25 | 0.712 | 0.742 |
| Hierarchical Transformer | Edge mode, width 32, dropout 0.45, weight decay 0.05, 350 epoch cap | 0.682 | 0.683 |
| Neural factorization machine | Rank 32, dropout 0.3, weight decay 0.1, five full data refits | 0.712 | 0.797 |
| ChemBERTa | Frozen mean plus CLS embeddings, regularized head | 0.691 | 0.722 |
| ChemBERTa | Fine tuned with randomized SMILES, 40 epoch cap | 0.669 | 0.672 |
| Neural factorization machine | Rank 32, feature mixup alpha 0.4, single 90% train fit | 0.712 | 0.769 |
| Neural factorization machine | Rank 32, batch 32, dropout 0.1, weight decay 0.01, single 90% train fit | 0.723 | 0.776 |
| Neural factorization machine | Rank 32, positive-class weight 1.5, five full data refits | 0.747 | 0.808 |
| Neural factorization machine | Rank 32, 4,096 graph-hash features, five full data refits | 0.745 | 0.812 |
| Neural oblivious-tree ensemble | 16 trees, depth 3, 512 selected features, 157 epoch checkpoint | 0.678 | 0.721 |
| Neural soft-tree ensemble | 16 trees, depth 3, 512 selected features, 141 epoch checkpoint | 0.686 | 0.743 |
| Neural factorization machine | Rank 32, five-fold OOF calibration, 2 epochs, five refits | 0.732 | 0.809 |
| Neural factorization machine | Rank 32, random-forest importance top 128 features, five refits | 0.708 | 0.754 |
| Neural factorization machine | Rank 32, chi-square top 128 features, five refits | 0.693 | 0.743 |
| Neural factorization machine | Rank 32, R-Drop alpha 0.1, five full data refits, validation threshold | 0.758 | 0.820 |
| Neural factorization machine | Rank 32, R-Drop, five-fold OOF, up to 300 epochs, validation-loss stopping, OOF threshold | 0.743 | 0.798 |
| Neural factorization machine | Rank 32, R-Drop, ten-fold OOF, four epochs, OOF threshold | 0.745 | 0.809 |
| Neural factorization machine | Rank 64, R-Drop, dropout 0.3, weight decay 0.1, ten-fold OOF | 0.756 | 0.822 |
| Neural ensemble | Five-model OOF stack, MLP width 8 meta-readout, OOF threshold | 0.767 | 0.815 |
| Neural ensemble | Three-repeat OOF stack, MLP width 16 meta-readout | 0.752 | 0.816 |
| Neural ensemble | OOF stack including the node Transformer, MLP width 8 | 0.741 | 0.815 |
| Fingerprint MLP | Width 256, dropout 0.5, ten-fold OOF, validation-loss stopping | 0.732 | 0.804 |
| Hierarchical Transformer | Node mode, width 32, five-fold OOF, 20 epochs | 0.704 | 0.760 |
| Neural factorization machine | Rank 128, R-Drop, dropout 0.3, weight decay 0.1, ten-fold OOF | 0.723 | 0.817 |
| DeepFM | Rank 16, MLP widths 128/32, dropout 0.4, weight decay 0.05, R-Drop, ten-fold OOF | 0.739 | 0.804 |
| Neural ensemble | 40% ten-fold R-Drop FM + 60% ten-fold DeepFM, OOF-selected | 0.747 | 0.810 |
| Relation-aware GIN | Width 64, three layers, dropout 0.4, 88 epochs, MPS | 0.691 | 0.730 |
| Neural factorization machine | Rank 32, R-Drop, twenty full-data seeds, four epochs, OOF threshold | 0.741 | 0.815 |
| Neural factorization machine | Morgan radius 2, 2,048 bits, R-Drop, ten-fold OOF | 0.702 | 0.757 |
| Neural factorization machine | AbstractGraph counts with fold-local TF-IDF, R-Drop, ten-fold OOF | 0.702 | 0.769 |
| Neural factorization machine | Rank 32, R-Drop count-transform sweep, raw counts selected | 0.734 | 0.817 |
| Neural factorization machine | Rank 32, R-Drop dropout sweep, dropout 0.3 selected | 0.723 | 0.803 |
| Neural factorization machine | Rank 32, R-Drop, five-fold OOF threshold, five refits | 0.739 | 0.809 |
| Neural factorization machine | Rank 32, R-Drop plus EMA decay 0.95, five refits | 0.723 | 0.810 |
| Neural factorization machine | Rank 32, SAM rho 0.02, five refits | 0.730 | 0.809 |
| Graph kernel reference | Random forest | 0.773 | 0.835 |

The highest exploratory neural-only result is a five-model stack of out-of-fold neural predictions with a width-8 MLP readout: it scores 0.767 accuracy and 0.815 ROC AUC with an OOF-selected threshold. This holdout has been consulted across many experiments, so treat the result as exploratory until verified on a fresh split. The rank-32 R-Drop FM scores 0.758 / 0.820 after five full-data refits, but its threshold came from one validation split and those refits include the validation examples. The original rank 32 FM scored 0.749. Five-fold out-of-fold calibrated R-Drop scored 0.739, so the holdout gain is split-sensitive. This is a neural classifier; no graph-kernel predictions are blended into it. The node Transformer remains the strongest direct graph model at 0.732. A fresh edge-mode run with the compact node settings stopped after 85 epochs; its minimum-validation-loss checkpoint came from epoch 16, while validation accuracy peaked at 0.754 in epoch 55, and test performance was 0.710 accuracy / 0.766 ROC AUC. Edge mode, the direct GIN, feature MLP, altered hash widths, mixup, chemistry descriptors, multi assay pretraining and class weighting did not improve accuracy. The graph kernel at 0.773 remains 0.7 percentage points above the exploratory OOF-calibrated neural stack at 0.767; the stack still needs confirmation on a fresh split. Additional trials did not improve the neural result: pairwise FM loss scored 0.725, co-occurrence initialized FM 0.734, a node Transformer and FM blend 0.743, Monte Carlo dropout 0.730, a feature set Transformer 0.719, a directed MPNN 0.688, and an FM with batch size 256 scored 0.728. The five fold calibrated FM refit scored 0.743, showing that the 0.749 holdout result is sensitive to the validation threshold. Further regularization did not close the gap: edge mode with stronger dropout and weight decay reached 0.721 validation accuracy and 0.682 test accuracy; a rank 32 FM with dropout 0.3 and weight decay 0.1 reached 0.783 validation accuracy but 0.712 test accuracy after five full data refits. ChemBERTa-77M-MLM frozen mean/CLS embeddings reached 0.691 test accuracy; fine tuning the encoder for 40 epochs with randomized SMILES reached 0.669. The FM with feature mixup (alpha 0.4) reached 0.772 validation accuracy and 0.712 test accuracy; a batch 32 FM reached 0.783 validation accuracy and 0.723 test accuracy. Positive-class weighting improved validation accuracy to 0.804 but reached 0.747 test accuracy after five refits; an all-neural blend of weighted and unweighted FMs reached 0.745. Increasing graph-hash width to 4,096 features scored 0.745 test accuracy, while 8,192 features reduced validation performance. A differentiable oblivious-tree ensemble trained for 157 epochs reached 0.750 validation accuracy but only 0.678 test accuracy. Out-of-fold threshold calibration selected the unweighted 2-epoch FM and reached 0.732 test accuracy. R-Drop with dropout 0.2 rose to 0.758 on the single holdout but fell to 0.739 under five-fold OOF calibration; adding EMA or SAM did not improve it. Count-transform and dropout sweeps scored 0.734 and 0.723 test accuracy. A more expressive soft-tree neural ensemble trained for up to 200 epochs reached 0.750 validation accuracy but only 0.686 test accuracy. Random-forest importance and chi-square feature selection overfit their inner validation results and scored 0.708 and 0.693 on test. These neural-only experiments did not exceed the prior rank 32 FM result or the kernel. Follow-up cross-validation found 0.745 test accuracy / 0.809 ROC AUC for the ten-fold R-Drop ensemble with the threshold set from OOF predictions; extending R-Drop training with validation-loss stopping reached 0.743 / 0.798, and twenty full-data seeds reached 0.741 / 0.815 at the OOF threshold. Label smoothing selected epoch 10 by OOF accuracy but its OOF-selected threshold transferred poorly to the holdout (0.710 accuracy, versus 0.747 at 0.5). Morgan fingerprints and fold-local TF-IDF inputs scored 0.702 test accuracy each. The more defensible cross-validated neural results remain below the 0.773 kernel reference. Increasing factor rank to 64 reached 0.756 but rank 128 fell to 0.723; a regularized DeepFM reached 0.739 and its OOF-selected neural-only blend reached 0.747. A wider, deeper relation-aware GIN trained for 88 epochs reached 0.691. The standalone fingerprint MLP reached 0.732 in ten-fold OOF evaluation. A five-fold node Transformer with fixed 20-epoch training reached 0.704; including it in the neural stack reduced test accuracy to 0.741. Repeated cross-validation of the strongest base models did not improve the OOF stack.

### Fresh five-fold edge-mode comparison

To compare edge mode with the kernel using identical stratified folds, the hierarchical Transformer was refit on each of five outer training folds. Each fit used an internal 10% validation split for early stopping. Configuration: edge tokens, local/global width 32, one Transformer layer at each level, dropout 0.3, batch size 64, up to 120 epochs, patience 20, and MPS. OOF probabilities were scored at threshold 0.5. The random-forest kernel row uses the same outer-fold predictions and threshold.

| Model | OOF accuracy | OOF ROC AUC |
| --- | ---: | ---: |
| Random-forest graph kernel | 0.744 | 0.812 |
| Hierarchical Transformer, edge mode | 0.700 | 0.733 |

Edge-mode fold accuracy / ROC AUC were 0.715 / 0.762, 0.671 / 0.699, 0.712 / 0.755, 0.708 / 0.747, and 0.694 / 0.735. Training took about 4.7 minutes total. The model overfit within folds and lagged the kernel by 4.4 percentage points in OOF accuracy and 0.079 ROC AUC. This fresh-fold run does not support switching the main experiment to edge mode.

### Longer training and regularization on fresh folds

These follow-up runs use the same five outer folds as the kernel reference (5-fold stratified CV, seed 77713). All reported neural predictions are neural-only.

| Model | Outer accuracy (inner-selected threshold) | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Random-forest graph kernel | 0.749 | 0.744 | 0.812 |
| Rank-32 R-Drop FM, dropout 0.4, weight decay 0.1, up to 64 epochs | 0.735 | 0.731 | 0.793 |
| Rank-32 R-Drop FM, nested choice of raw / square-root / log / presence counts and epoch | 0.734 | 0.736 | 0.784 |
| Low-rank cross network plus MLP, log-count inputs, up to 150 epochs | 0.683 | 0.668 | 0.749 |
| Nested neural-only ensemble of rank-32 FM, rank-64 FM, and DeepFM | 0.736 | 0.735 | 0.796 |

The longer R-Drop run selected only 4–16 epochs in the five folds, despite a 64-epoch cap. The low-rank cross network selected epoch 1 in every fold. Neither longer training nor stronger regularization improved on the kernel. The count transform choices varied across folds and did not improve accuracy.

The earlier fresh-fold neural stack estimate of 0.745 nested accuracy should be treated as optimistic: its base OOF predictions used different fold assignments from the meta-model evaluation, so some base models used to create meta-training features may have trained on labels in the meta-test fold. A fully nested FM/DeepFM ensemble, with all base fits isolated inside each outer fold, scored 0.736 nested accuracy and 0.735 OOF accuracy at 0.5. The kernel remains ahead.

Additional regularization variants on the same outer folds also fell short:

| Model | Outer accuracy (inner-selected threshold) | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Rank-32 R-Drop FM, learning rate 2e-4, dropout 0.35, weight decay 0.05 | 0.721 | 0.717 | 0.783 |
| Supervised-contrastive fingerprint MLP, dropout 0.45 | 0.718 | 0.718 | 0.759 |

The lower-learning-rate FM selected checkpoints at epochs 3–9; the contrastive MLP selected epochs 1–2 despite a 120-epoch cap. Their thresholds were selected on an internal 10% validation split. Across the tested FM, MLP, and cross-network variants, longer caps did not translate into longer useful training or better outer-fold performance.

### Pretrained tabular Transformer

I also evaluated the Prior Labs TabPFN v2 transformer as a neural tabular model for this small dataset. TabPFN v2 was run on MPS with 16 estimators and 500 features per fold; features were selected using either variance (unsupervised) or chi-square against that fold's training labels. The chi-square selection was fit separately inside each outer fold. OOF metrics use threshold 0.5 and the same five outer folds as the kernel.

| Model | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: |
| Random-forest graph kernel | 0.744 | 0.812 |
| TabPFN v2, top 500 by variance | 0.718 | 0.780 |
| TabPFN v2, top 500 by chi-square | 0.722 | 0.779 |

Using all 2,048 features with the v2 pretraining-limit guard disabled ran for about five minutes under high system memory pressure and was interrupted before a score was produced. I excluded it from the comparison. TabPFN v2 is a pretrained neural Transformer; its code and model weights have separate licenses, so this internal benchmark used the v2 weights under the Prior Labs License and did not redistribute weights. See the [TabPFN v2 model card](https://huggingface.co/Prior-Labs/TabPFN-v2-clf), [v2 feature-limit configuration](https://github.com/PriorLabs/TabPFN/blob/main/src/tabpfn/inference_config.py), and [v2 weight license](https://huggingface.co/Prior-Labs/TabPFN-v2-clf/blob/main/LICENSE.txt).

Follow-up TabPFN checks did not change the conclusion. On one outer fold, splitting the chi-square ranking into four disjoint 500-feature blocks and averaging four v2 models (8 estimators each) scored 0.662 accuracy / 0.784 ROC AUC at threshold 0.5; the blocks ranked 1–500, 501–1000, 1001–1500, and 1501–2000 scored 0.743, 0.638, 0.643, and 0.641 accuracy individually. This was a one-fold diagnostic and was not advanced to full CV. A separate TabPFN v2 fit with built-in accuracy threshold tuning did not finish within five minutes and was interrupted without a score.


### November 2026 regularization trials

These neural-only trials use the same five outer folds as the kernel reference (stratified CV, seed 77713) and the aligned `X_fresh` / `y_fresh` arrays. The random-forest graph kernel scored 0.749 mean outer accuracy with inner-selected thresholds, 0.744 OOF accuracy at 0.5, and 0.812 OOF ROC AUC.

| Neural experiment | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Rank-32 R-Drop FM, 10% input-feature masking | — | 0.715 | 0.778 |
| Rank-32 R-Drop FM, 25% input-feature masking | — | 0.708 | 0.762 |
| Rank-32 R-Drop FM, factor-vector weight decay 0.03 | — | 0.705 | 0.758 |
| Rank-32 R-Drop FM, factor-vector weight decay 0.10 | — | 0.705 | 0.758 |
| Rank-32 R-Drop FM, factor-vector weight decay 0.30 | — | 0.705 | 0.758 |
| Rank-32 focal-loss FM, gamma selected from 0.5 / 1 / 2 | 0.715 | 0.721 | 0.778 |

For feature masking and factor-specific decay, each outer training fold used a stratified 12% validation split to select the checkpoint by validation loss, then refit on the full outer training fold for the selected epoch. The 80-epoch cap was not useful: selected epochs were 1–4. The focal-loss run used five-fold inner OOF predictions to choose gamma, epoch (2–32), and threshold, then refit each outer fold. Its selected epochs were 4–16. None of these methods improved on the kernel. The weaker results from feature masking and factor-specific decay are exploratory single-seed OOF comparisons; the focal-loss result is fully nested.


### Factor-interaction dropout

I tested dropout on the factorization machine's pairwise interaction contributions, in addition to its existing input dropout. For each outer fold, five inner folds selected interaction dropout (0.2 or 0.5), epoch (2, 4, 8, 16, or 24), and a classification threshold. Each selected model was refit on the full outer training fold. This keeps the model neural-only and evaluates selection on held-out outer folds.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Rank-32 R-Drop FM with interaction dropout | 0.719 | 0.724 | 0.776 |
| Random-forest graph kernel | 0.749 | 0.744 | 0.812 |

The selected interaction dropout varied by fold (0.2 in three folds and 0.5 in two), with checkpoints at epochs 8 or 16. This did not improve generalization over the existing neural FM results.


### Neural-only self-training with auxiliary molecules

I used 6,480 auxiliary molecules without exact target-set graph-hash overlap as unlabeled data. Inside each outer fold, two rank-32 neural FM teachers were trained on that fold's training labels; their mean predictions supplied pseudo-labels for auxiliary examples with confidence at least 0.85 or 0.95. Inner five-fold predictions selected the confidence cutoff, training epoch (4 or 8), and decision threshold. The final neural student was refit on the full outer training fold plus up to 2,000 pseudo-labeled auxiliary examples, with pseudo-label loss weighted 0.25.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Neural FM self-training | 0.725 | 0.728 | 0.775 |
| Random-forest graph kernel | 0.749 | 0.744 | 0.812 |

The selected confidence cutoff was 0.85 or 0.95 depending on the fold. This neural-only self-training setup did not improve on supervised FM training; the pseudo-labels from auxiliary assay molecules did not transfer reliably enough to close the kernel gap.


### Standalone rank-64 R-Drop FM on the kernel folds

The rank-64 R-Drop factorization machine had scored well in an earlier 10-fold run, so I evaluated it alone against the kernel's five outer folds (stratified CV, seed 77713). Each outer fold used five inner folds to select a checkpoint from epochs 4, 8, 16, 32, 48, and 64 and a decision threshold. The outer refit averaged three neural seeds at the selected epoch.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Rank-64 R-Drop FM, three-seed outer refit | 0.725 | 0.731 | 0.785 |
| Random-forest graph kernel | 0.749 | 0.744 | 0.812 |

Selected checkpoints were epochs 8–32. The earlier rank-64 result did not transfer to these matched outer folds; increasing factor rank and seed averaging did not close the gap.


### Nested positive-class weighting

A single holdout experiment had suggested that weighting positives might help, so I retested it on the kernel's five outer folds (stratified CV, seed 77713). Five inner folds selected positive-class weight (1.0, 1.25, or 1.5), checkpoint epoch (2, 4, 8, 16, or 32), and decision threshold. Each outer refit averaged three neural FM seeds.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Rank-32 R-Drop FM with inner-selected class weighting | 0.731 | 0.725 | 0.780 |
| Random-forest graph kernel | 0.749 | 0.744 | 0.812 |

Four of the five inner folds selected positive weight 1.5, but outer-fold performance remained below the unweighted neural FM and the kernel. The earlier holdout gain did not transfer to this nested evaluation.


### Late-checkpoint averaging (SWA) and snapshot ensembles

To use later training while reducing sensitivity to one epoch, I trained the rank-32 R-Drop FM through epoch 32 with cosine learning-rate decay after epoch 8. Five inner folds selected a raw checkpoint or a late-window average and a threshold. Each outer refit averaged either three or eight independently initialized neural models. I also evaluated ensembles that average predictions across checkpoints instead of weights.

| Neural model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| SWA, three seeds | 0.738 | 0.736 | 0.792 |
| SWA, eight seeds | 0.739 | 0.736 | 0.797 |
| Late-checkpoint probability ensemble, three seeds | 0.730 | 0.734 | 0.788 |
| Random-forest graph kernel | 0.749 | 0.744 | 0.812 |

Inner selection chose a raw epoch-8 checkpoint in two folds and weight-averaged windows in the other three. Increasing the outer ensemble from three to eight seeds improved nested accuracy by about 0.2 percentage points, but did not improve accuracy at threshold 0.5 enough to close the gap. Snapshot probability averaging scored below weight averaging.

These five outer folds have been reused across many development experiments. Treat the model-to-model differences as exploratory; a neural candidate must beat the kernel again on an untouched split or repeated independent evaluation before claiming a reliable gain.


### Fresh-seed comparison of SWA and the graph-hash kernel

Because the original five outer folds had been reused during model development, I ran another matched five-fold evaluation with stratification seed 90817. Both models used identical outer folds. The kernel used `RandomForestClassifier(n_estimators=300, random_state=0)`; its threshold was selected from inner five-fold OOF predictions. The rank-32 SWA procedure used inner five-fold predictions to select a checkpoint window and threshold, then refit an eight-seed ensemble on each full outer training fold.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Random-forest graph kernel | 0.756 | 0.754 | 0.820 |
| Rank-32 R-Drop FM with SWA, eight seeds | 0.738 | 0.741 | 0.803 |

The fresh folds confirm that the current SWA method does not beat the kernel. This is a new split seed, but it is still development data because model-family choices were made after inspecting earlier CV results; retain a separate final holdout for a definitive claim.


### Fold-local frequency shrinkage on a fresh CV seed

Nearly half the fingerprint dimensions appeared in three or fewer target molecules. I tested whether shrinking rare-feature inputs could reduce memorization. On stratified five-fold CV with seed 92357, inner five-fold predictions selected a training-only weight for each feature, `df / (df + alpha)`, with alpha in {-1 (unweighted), 0 (mask unseen features), 2, 10, or 30}. Inner folds also selected checkpoint epoch (2, 4, 8, 16, or 32) and threshold. Each selected model was refit as a five-seed neural FM ensemble. A 300-tree random forest was evaluated on the same folds with its threshold selected inside each outer training fold.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Frequency-shrunk rank-32 R-Drop FM | 0.734 | 0.736 | 0.792 |
| Random-forest graph kernel | 0.746 | 0.744 | 0.813 |

The selected alpha varied across folds (-1, 2, or 30), and the neural model remained below the kernel. Frequency-based shrinkage did not improve generalization reliably.


### Fresh-seed DeepFM comparison

To test a neural architecture with an explicit higher-order deep tower, I evaluated the rank-16 DeepFM on stratified five-fold CV with seed 93241. It used dropout 0.4, AdamW learning rate 5e-4, weight decay 0.05, batch size 128, and R-Drop. Inner five-fold predictions selected the epoch (2, 4, 8, 16, or 32) and threshold; each outer model averaged five neural seeds. The 300-tree random-forest kernel used identical outer folds and an inner-selected threshold.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| DeepFM with five-seed outer refit | 0.731 | 0.738 | 0.792 |
| Random-forest graph kernel | 0.749 | 0.743 | 0.814 |

The inner folds selected epochs 16 or 32, but the longer DeepFM training did not improve outer-fold performance.


### Generalized cross-entropy on a fresh CV seed

To reduce the effect of potentially noisy assay labels, I evaluated generalized cross-entropy (GCE) with q in {0.3, 0.5, 0.7, 1.0}. Five inner folds selected q, checkpoint epoch (2, 4, 8, 16, or 32), and threshold. Each outer refit averaged five rank-32 R-Drop FM seeds. The matched 300-tree random-forest kernel used the same five outer folds, stratified seed 94531, and an inner-selected threshold.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Rank-32 R-Drop FM with GCE | 0.732 | 0.740 | 0.796 |
| Random-forest graph kernel | 0.743 | 0.746 | 0.813 |

The selected q and epoch varied by fold (q = 0.3–1.0; epochs 4–16). GCE did not produce a stable gain over BCE or close the kernel gap.


### Co-teaching for assay-label noise

I evaluated co-teaching on stratified five-fold CV with seed 95639. Two rank-32 neural FMs exchanged their small-loss examples; the inner folds selected a forget rate (0.1, 0.2, or 0.3), checkpoint epoch (4, 8, 16, or 32), and threshold. Each outer refit averaged three independently initialized co-teaching pairs. The 300-tree random-forest kernel used the same folds and an inner-selected threshold.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Neural FM co-teaching | 0.728 | 0.728 | 0.793 |
| Random-forest graph kernel | 0.743 | 0.743 | 0.811 |

The selected forget rates were 0.1–0.2 and epochs were 4–16. Co-teaching did not improve generalization over the kernel or the stronger supervised neural FM runs.


### Neural random-subspace ensemble on a fresh CV seed

I tested whether an ensemble of neural models trained on random hash-feature subsets would reduce correlated overfitting. On stratified five-fold CV with seed 96797, inner five-fold predictions selected feature retention (50%, 75%, or 100%), checkpoint epoch (2, 4, 8, or 16), and threshold. The final outer model averaged eight rank-32 R-Drop FM subspaces. The 300-tree random-forest kernel used identical folds and an inner-selected threshold.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Neural random-subspace ensemble | 0.732 | 0.737 | 0.797 |
| Random-forest graph kernel | 0.750 | 0.749 | 0.816 |

The inner folds selected 100% of features in three folds and 75% in two. Random subspaces did not improve neural generalization.


### 4,096-bit hash width (nbits=12) on matched folds

The earlier holdout experiment suggested that 4,096 graph-hash features might help, so I regenerated features at `nbits=12` and evaluated on stratified five-fold CV with seed 98977. I verified that regenerating `nbits=11` features exactly reproduced the existing 2,048-column matrix. Inner five-fold predictions selected the rank-32 R-Drop FM checkpoint epoch and threshold; each outer refit averaged five neural seeds. Both random-forest widths used the same outer and inner folds and 300 trees.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Rank-32 R-Drop FM, 4,096 bits (`nbits=12`) | 0.739 | 0.740 | 0.796 |
| Random-forest graph hash, 4,096 bits | 0.745 | 0.746 | 0.808 |
| Random-forest graph hash, 2,048 bits | 0.745 | 0.752 | 0.809 |

Widening the representation did not close the neural-kernel gap.


### Sixteen-layer residual MLP on fresh folds

I evaluated an explicit 16-hidden-layer residual MLP on stratified five-fold CV with seed 100003. It has eight width-128 residual units with two dense layers apiece, LayerNorm, GELU, 0.3 dropout, up to 0.15 stochastic depth, 0.2 input dropout, and 0.1 residual layer scaling. Training used AdamW (learning rate 5e-4, weight decay 0.05), batch size 128, R-Drop, gradient clipping, cosine decay, and up to 64 epochs. Inner five-fold predictions selected raw versus log1p counts, epoch, and threshold; each outer refit averaged three neural seeds. The kernel used the same folds and inner-selected threshold.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Sixteen-layer residual MLP | 0.730 | 0.728 | 0.759 |
| Random-forest graph kernel | 0.733 | 0.739 | 0.804 |

The selected epochs were 16, 16, 64, 16, and 4. Only one fold selected the 64-epoch cap; the others selected substantially earlier checkpoints. The deep MLP came close in nested accuracy but had clearly weaker ranking performance and lower OOF accuracy.


### Frequency-weighted factor-parameter penalty

On stratified five-fold CV with seed 97859, I added an explicit penalty to the FM's factor rows and linear coefficients. Penalty weights were computed from feature document frequency within each fit fold, with rarer features receiving stronger shrinkage. Inner five-fold predictions selected penalty strength (0, 1e-4, 5e-4, or 2e-3), epoch (2, 4, 8, or 16), and threshold; each outer refit averaged five neural seeds. The 300-tree random-forest kernel used the same folds.

| Model | Nested outer accuracy | OOF accuracy at 0.5 | OOF ROC AUC |
| --- | ---: | ---: | ---: |
| Frequency-penalized rank-32 R-Drop FM | 0.731 | 0.738 | 0.795 |
| Random-forest graph kernel | 0.751 | 0.749 | 0.813 |

Frequency-weighted parameter shrinkage did not improve neural generalization.


### Sixteen-layer residual MLP extended to 128 epochs

I extended the same residual architecture to a maximum of 128 epochs on stratified five-fold CV with seed 101111. It uses eight width-128 two-layer residual units (16 hidden Linear layers), LayerNorm, GELU, dropout 0.3, stochastic depth up to 0.15, input dropout 0.2, and residual scaling 0.1. Training used AdamW (learning rate 5e-4, weight decay 0.05), batch size 128, R-Drop, gradient clipping, and cosine decay. Inner five-fold predictions selected raw versus log1p counts, checkpoint epoch, and threshold; each outer refit averaged three independently initialized neural models. The 300-tree random-forest kernel used the same folds and an inner-selected threshold.

The longer cap did not close the gap. The residual MLP reached 0.7371 nested outer accuracy, 0.7354 OOF accuracy at threshold 0.5, and 0.7575 OOF ROC AUC. The matched kernel reached 0.7528, 0.7502, and 0.8204, respectively. The MLP's nested accuracy improved modestly over its earlier 64-epoch run (0.7301 on seed 100003), but remained 1.57 percentage points below the kernel on this seed. Inner selection chose epochs 8, 64, 16, 8, and 16 across the five folds; it did not select epochs beyond 64 despite the 128-epoch cap.

| Fold | MLP selected epoch | MLP outer accuracy | MLP ROC AUC | Kernel outer accuracy | Kernel ROC AUC |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 8 | 0.7691 | 0.8119 | 0.7756 | 0.8362 |
| 2 | 64 | 0.7102 | 0.7802 | 0.7560 | 0.8433 |
| 3 | 16 | 0.7233 | 0.7627 | 0.7320 | 0.7908 |
| 4 | 8 | 0.7407 | 0.7575 | 0.7233 | 0.8037 |
| 5 | 16 | 0.7424 | 0.7697 | 0.7773 | 0.8292 |

This is evidence that extending training alone is insufficient to beat the kernel. The next deep-model comparison should target regularization and optimization choices under the same nested protocol, while keeping the outer folds untouched during inner selection.


### Residual MLP normalization and initialization sweep

I compared normalization and initialization choices on stratified five-fold CV with seed 110003. Each configuration kept the same 16-hidden-layer residual MLP, R-Drop training, AdamW settings, batch size, cosine schedule, and three-seed outer refit. Inner five-fold predictions selected raw versus log1p counts, checkpoint epoch (8, 16, 32, or 64), and threshold. The 300-tree random-forest kernel used the same folds and an inner-selected threshold. This sweep took 43.42 minutes.

| Configuration | Normalization | Linear initialization | Nested accuracy | OOF accuracy at 0.5 | OOF ROC AUC | Selected epochs by fold |
| --- | --- | --- | ---: | ---: | ---: | --- |
| LayerNorm default | LayerNorm | PyTorch default | 0.7315 | 0.7341 | 0.7599 | 16, 8, 32, 8, 8 |
| RMSNorm default | RMSNorm | PyTorch default | 0.7310 | 0.7271 | 0.7467 | 64, 32, 8, 16, 32 |
| BatchNorm default | BatchNorm1d | PyTorch default | 0.7354 | 0.7337 | 0.7786 | 16, 8, 32, 8, 64 |
| LayerNorm Xavier | LayerNorm | Xavier uniform | 0.7393 | 0.7406 | 0.7753 | 16, 32, 16, 8, 16 |
| LayerNorm orthogonal | LayerNorm | Orthogonal | 0.7371 | 0.7411 | 0.7767 | 8, 16, 8, 8, 16 |
| Random-forest kernel | — | — | 0.7419 | 0.7467 | 0.8146 | — |

Xavier initialization gave the best nested accuracy among these neural configurations, but it remained 0.26 percentage points below the kernel; its AUC was lower by 0.0393. BatchNorm had the strongest neural AUC, still 0.0360 below the kernel. RMSNorm performed worse than the LayerNorm default. No neural variant beat the kernel overall, and selected epochs remained mostly 8–32, so simply allowing longer training has not addressed the gap.

The normalization and initialization configurations were compared across the same outer folds rather than selected inside the inner CV. Treat the ranking among neural variants as exploratory; confirm any candidate on a fresh split before selecting it. The next experiment should put regularization strength and dropout selection inside the inner folds and retain the matched kernel for each outer split.


### Inner-selected regularization for the residual MLP

On fresh stratified five-fold CV (seed 111007), I moved regularization selection inside the inner folds. The candidate set included the LayerNorm/Xavier baseline (dropout 0.3, input dropout 0.2, stochastic depth 0.15, weight decay 0.05, R-Drop 0.1), stronger dropout (0.5 / 0.3 / 0.25), higher weight decay (0.2), stronger R-Drop (0.3), and a BatchNorm/default-init baseline. Inner five-fold predictions jointly selected the candidate, raw versus log1p counts, epoch (8, 16, 32, or 64), and threshold. Each outer refit averaged three neural seeds. The 300-tree random-forest kernel used the same folds and inner threshold selection. The experiment took 44.51 minutes.

| Fold | Selected neural setting | Epoch | Neural outer accuracy | Neural ROC AUC | Kernel outer accuracy | Kernel ROC AUC |
| ---: | --- | ---: | ---: | ---: | ---: | ---: |
| 1 | Xavier, stronger R-Drop (0.3) | 8 | 0.7320 | 0.7585 | 0.7277 | 0.8022 |
| 2 | Xavier baseline | 8 | 0.7342 | 0.7913 | 0.7603 | 0.8203 |
| 3 | Xavier, weight decay 0.2 | 8 | 0.7320 | 0.7774 | 0.7320 | 0.7940 |
| 4 | Xavier baseline | 8 | 0.7342 | 0.8117 | 0.7560 | 0.8385 |
| 5 | Xavier, stronger dropout | 16 | 0.7751 | 0.8190 | 0.7598 | 0.8176 |

Across folds, the inner-selected neural model reached 0.7415 nested accuracy, 0.7402 OOF accuracy at 0.5, and 0.7922 OOF ROC AUC. The kernel reached 0.7472, 0.7546, and 0.8139. Neural accuracy was higher on folds 1 and 5, tied on fold 3, and lower on folds 2 and 4; its pooled accuracy and AUC remained below the kernel. The selected settings varied, but training still stopped at epochs 8 or 16. Stronger regularization and longer training did not close the overall gap.


### Nested neural-only ensemble of residual model variants

I tested whether averaging diverse residual MLPs reduces variance on stratified five-fold CV with seed 112013. Inner five-fold predictions evaluated five neural configurations: LayerNorm/Xavier baseline, stronger dropout, higher weight decay, stronger R-Drop, and BatchNorm/default initialization. For each configuration, inner predictions selected raw versus log1p counts and checkpoint epoch (8, 16, 32, or 64). I ranked the configurations by inner accuracy and compared equal-weight ensembles of the top 1, 2, 3, or all 5, with the threshold also selected inside the inner folds. Each selected member averaged three independent outer-refit seeds. The matched 300-tree random-forest kernel used the same folds and inner threshold selection. The run took 52.93 minutes.

| Fold | Selected ensemble size | Selected neural members | Neural outer accuracy | Neural ROC AUC | Kernel outer accuracy | Kernel ROC AUC |
| ---: | ---: | --- | ---: | ---: | ---: | ---: |
| 1 | 3 | Dropout, R-Drop, weight decay | 0.7342 | 0.7723 | 0.7495 | 0.8052 |
| 2 | 5 | All five configurations | 0.7560 | 0.8160 | 0.7473 | 0.8206 |
| 3 | 3 | Dropout, R-Drop, BatchNorm | 0.7298 | 0.7707 | 0.7190 | 0.7912 |
| 4 | 5 | All five configurations | 0.7342 | 0.7574 | 0.7342 | 0.7972 |
| 5 | 2 | BatchNorm, R-Drop | 0.7445 | 0.8243 | 0.7838 | 0.8566 |

The neural ensemble reached 0.7397 nested outer accuracy, 0.7437 OOF accuracy at threshold 0.5, and 0.7784 OOF ROC AUC. The kernel reached 0.7468, 0.7459, and 0.8141. The ensemble was ahead in nested accuracy on folds 2 and 3, tied on fold 4, and behind on folds 1 and 5; it did not beat the kernel overall, especially on ranking quality (AUC). This neural-only ensemble reduces the fixed-0.5 accuracy gap, but does not establish superiority over the kernel.


### Residual MLP at 2,048 versus 4,096 hash bits

I compared graph-hash widths inside nested five-fold CV with seed 113009. The 16-hidden-layer residual MLP used LayerNorm, Xavier initialization, dropout 0.3, input dropout 0.2, stochastic depth 0.15, AdamW weight decay 0.05, R-Drop 0.1, batch size 128, and cosine decay. Inner five-fold predictions selected hash width (2,048 or 4,096), raw versus log1p counts, epoch (8, 16, 32, or 64), and threshold. The random-forest kernel separately selected its hash width and threshold on the same inner folds. Each neural outer refit averaged three seeds. The run took 23.49 minutes.

| Fold | MLP width | MLP input | MLP epoch | MLP outer accuracy | MLP ROC AUC | RF width | RF outer accuracy | RF ROC AUC |
| ---: | ---: | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 4,096 | Raw | 8 | 0.7451 | 0.8175 | 4,096 | 0.7843 | 0.8481 |
| 2 | 4,096 | Raw | 64 | 0.7211 | 0.7758 | 2,048 | 0.7320 | 0.8089 |
| 3 | 4,096 | Raw | 64 | 0.7451 | 0.7859 | 4,096 | 0.7603 | 0.8388 |
| 4 | 4,096 | log1p | 16 | 0.7211 | 0.7509 | 4,096 | 0.7037 | 0.7732 |
| 5 | 4,096 | Raw | 16 | 0.7576 | 0.7790 | 2,048 | 0.7555 | 0.8240 |

The inner folds selected 4,096 bits for the MLP in all five folds, but this did not improve the overall result: the MLP reached 0.7380 nested accuracy, 0.7384 OOF accuracy at threshold 0.5, and 0.7627 OOF ROC AUC. The width-selected kernel reached 0.7472, 0.7437, and 0.8187. Although the MLP exceeded the kernel's thresholded accuracy on folds 4 and 5, its overall accuracy and ranking quality remained lower. Increasing hash width alone does not solve the generalization gap.
